In [1]:
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from torch_geometric.utils import dense_to_sparse
import os
import pandas as pd
import numpy as np
import torch
import pickle
from sklearn.preprocessing import StandardScaler
from torch_geometric.utils import dense_to_sparse



def draw_picture(adj_matrix, degree, season, threshold, type):
    # 度分布直方图
    plt.figure(figsize=(8, 5))
    plt.hist(degree, bins=30, edgecolor='k')
    plt.xlabel("Degree (Number of Connections)")
    plt.ylabel("Number of Nodes")
    plt.title(f"Degree Distribution (Threshold = {threshold})")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"./output/{season}_{type}_degree_distribution.png")
    plt.close()

    # 邻接矩阵热力图
    plt.figure(figsize=(10, 8))
    sns.heatmap(adj_matrix, cmap='Blues')
    plt.title(f"Adjacency Matrix Heatmap (Threshold = {threshold})")
    plt.xlabel("Node Index")
    plt.ylabel("Node Index")
    plt.tight_layout()
    plt.savefig(f"./output/{season}_{type}_adjacency_matrix_heatmap.png")
    plt.close()


def build_topk_graph_from_data(feather_path, codes, train_date_list, season, type, top_k=100, threshold=1000, output_dir="./output"):
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 读取数据
    graph_data = pd.read_feather(feather_path)
    graph_data.set_index('index', inplace=True)
    graph_data.fillna(0, inplace=True)

    # 将索引转为 datetime 类型（如果还不是）
    graph_data.index = pd.to_datetime(graph_data.index, format="%Y%m%d")

    # 找出 train_date_list 中最晚的一天，并转为 datetime 类型
    max_date = pd.to_datetime(max(train_date_list), format="%Y%m%d")
    min_date = pd.to_datetime(min(train_date_list), format="%Y%m%d")

    # 保留索引 <= max_date 的数据
    graph_data = graph_data[(graph_data.index >= min_date) & (graph_data.index <= max_date)]

    cols_to_keep = [col for col in codes if col in graph_data.columns]
    graph_data = graph_data[cols_to_keep]
    graph_data.to_feather(os.path.join(output_dir, f"{season}_{type}.fea"))

    # 2. 获取节点 ID
    node_ids = graph_data.columns.tolist()

    # 3. 标准化每列特征
    scaler = StandardScaler()
    graph_data_std = pd.DataFrame(scaler.fit_transform(graph_data), index=graph_data.index, columns=node_ids)

    # 4. 构建相关系数矩阵
    graph_corr_matrix = graph_data_std.corr().abs()
    np.fill_diagonal(graph_corr_matrix.values, 0)

    # 假设已有：graph_corr_matrix, top_k, edge_limit_per_node
    graph_corr_matrix_values = graph_corr_matrix.values
    triu_indices = np.triu_indices_from(graph_corr_matrix_values, k=1)
    correlations = graph_corr_matrix_values[triu_indices]

    # 获取 top_k 的候选边（按值排序）
    sorted_indices = np.argsort(correlations)[::-1]  # 从大到小
    top_pairs = list(zip(triu_indices[0][sorted_indices], triu_indices[1][sorted_indices]))

    # 初始化邻接矩阵和节点边计数
    adj_matrix = np.zeros_like(graph_corr_matrix_values, dtype=int)
    edge_count = np.zeros(graph_corr_matrix.shape[0], dtype=int)
    invalid_nodes = set()
    selected_edges = 0

    for i, j in top_pairs:
        if selected_edges >= top_k:
            break
        if i in invalid_nodes or j in invalid_nodes:
            continue
        if edge_count[i] >= threshold or edge_count[j] >= threshold:
            # 超过限制，将节点标记为无效并删除相关边
            invalid_nodes.update([i] if edge_count[i] >= threshold else [])
            invalid_nodes.update([j] if edge_count[j] >= threshold else [])
            # 删除所有与无效节点相关的边
            adj_matrix[i, :] = 0
            adj_matrix[:, i] = 0
            adj_matrix[j, :] = 0
            adj_matrix[:, j] = 0
            edge_count[i] = 0
            edge_count[j] = 0
            selected_edges = np.sum(adj_matrix) // 2
            continue
        # 添加该边
        adj_matrix[i, j] = 1
        adj_matrix[j, i] = 1
        edge_count[i] += 1
        edge_count[j] += 1
        selected_edges += 1

    # 移除自环
    np.fill_diagonal(adj_matrix, 0)

    # 6. 可视化分析图
    degree = adj_matrix.sum(axis=1)
    draw_picture(adj_matrix, degree, season, top_k, type)

    # 7. 稀疏边列表
    edge_index = dense_to_sparse(torch.tensor(adj_matrix))[0]
    print(f"[{season}] Edge index shape: {edge_index.shape}")

    # 8. 保存图数据
    graph_data_dict = {
        "edge_index": edge_index,
        "node_ids": node_ids
    }
    with open(os.path.join(output_dir, f"{season}_{type}_graph.pkl"), "wb") as f:
        pickle.dump(graph_data_dict, f)

    return edge_index, node_ids


def merge_edge_indices(edge_index1, edge_index2):
    # 1. 拼接两个 edge_index
    combined = torch.cat([edge_index1, edge_index2], dim=1)  # shape: (2, m + n)

    # 2. 转置为 (m+n, 2) 并去重（注意不排序，以保留边的方向）
    unique_edges = torch.unique(combined.T, dim=0).T  # shape: (2, num_unique_edges)

    print(f"edge_index1 shape: {edge_index1.shape}, edge_index2 shape: {edge_index2.shape}")
    print(f"Combined edge_index shape: {combined.shape}, Unique edge_index shape: {unique_edges.shape}")

    return unique_edges



def compute_edge_overlap(edge_index1, edge_index2):
    # 排序每条边（无向图：将[i, j] 与 [j, i] 视为同一条边）
    e1 = torch.sort(edge_index1, dim=0)[0].T  # shape: (m1, 2)
    e2 = torch.sort(edge_index2, dim=0)[0].T  # shape: (m2, 2)

    # 转为集合比较
    edges1_set = set(map(tuple, e1.tolist()))
    edges2_set = set(map(tuple, e2.tolist()))

    intersection = edges1_set & edges2_set
    union = edges1_set | edges2_set

    overlap_ratio = len(intersection) / len(union) if union else 0.0

    return overlap_ratio, len(intersection), len(union)


In [4]:
# 读取 pkl 文件
with open("/home/user96/GCN_v2_0.2/output/2023q1_subgraph_output.pkl", "rb") as f:
    graph_banchmark = pickle.load(f)

edge_banchmark = graph_banchmark["edge_index"]  # 图边 (Tensor)
node_banchmark = graph_banchmark["node_ids"]      # 股票 ID 列表


train_date_list = ['20240601', "20241231"]  # 训练日期列表
season = '2029q4'  # 季度
top_k = 160000  # 前 k 个边
threshold = 900  # 每个节点的最大边数限制
# threshold = 0.2

edge_index1, node_ids1 = build_topk_graph_from_data(
        feather_path='/home/user96/label_data1/can_trade_amt1.fea',
        codes=node_banchmark,
        train_date_list=train_date_list,
        type='liquid',
        season=season,
        top_k=top_k,
        threshold=threshold,
    )

# edge_index2, node_ids2 = build_topk_graph_from_data(
#         feather_path='/home/user96/label_data1/label1.fea',
#         codes=node_banchmark,
#         train_date_list=train_date_list,
#         type='label',
#         season=season,
#         top_k=top_k,
#         threshold=threshold,
#     )


# assert set(node_ids1) == set(node_ids2), "Node sets do not match!"
# assert node_ids1 == node_ids2, "Node order does not match!"

# edge_index = merge_edge_indices(edge_index1, edge_index2)

edge_index = edge_index1

[2029q4] Edge index shape: torch.Size([2, 320000])


In [5]:


overlap, num_common, num_total = compute_edge_overlap(edge_index, edge_banchmark)
print(f"Benchmark edge index shape: {edge_banchmark.shape}")
print(f"Current edge index shape: {edge_index.shape}")
print(f"重合度: {overlap* 100:.2f}%  |  交集边数: {num_common}  |  并集边数: {num_total}")
print(f"覆盖banchmark : {num_common / edge_banchmark.shape[1] * 200:.2f}%")

Benchmark edge index shape: torch.Size([2, 280538])
Current edge index shape: torch.Size([2, 320000])
重合度: 11.83%  |  交集边数: 31754  |  并集边数: 268515
覆盖banchmark : 22.64%
